In [ ]:
# Install nltk (usually pre-installed)
import nltk
nltk.download('stopwords')
nltk.download('punkt')

# Import libraries
import pandas as pd
import numpy as np
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
pip install -U datasets huggingface_hub

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load dataset using API
dataset = load_dataset("stanfordnlp/imdb")

# Convert training split to DataFrame
df = pd.DataFrame(dataset['train'])

# Convert label numbers to text
df['sentiment'] = df['label'].map({0: 'negative', 1: 'positive'})

# Keep required columns only
df = df[['text', 'sentiment']]

# Rename column to match rest of code
df.rename(columns={'text': 'review'}, inplace=True)

df.head()


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

,review,sentiment
0,I rented I AM CURIOUS-YELLOW from my video sto...,negative
1,"""I Am Curious: Yellow"" is a risible and preten...",negative
2,If only to avoid making this type of film in t...,negative
3,This film was probably inspired by Godard's Ma...,negative
4,"Oh, brother...after hearing about this ridicul...",negative


In [ ]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})


In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_review'] = df['review'].apply(clean_text)


In [ ]:
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
cv = CountVectorizer(max_features=5000)

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)


In [ ]:
model = MultinomialNB()
model.fit(X_train_cv, y_train)


MultinomialNB()

In [ ]:
y_pred = model.predict(X_test_cv)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.8486
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      2515
           1       0.85      0.84      0.85      2485

    accuracy                           0.85      5000
   macro avg       0.85      0.85      0.85      5000
weighted avg       0.85      0.85      0.85      5000



In [ ]:
def predict_sentiment(review):
    review = clean_text(review)
    vector = cv.transform([review])
    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "Positive 😊"
    else:
        return "Negative 😞"

# Test
predict_sentiment("The product quality is amazing and worth the price")


'Positive 😊'

In [ ]:
def predict_sentiment(user_review):
    # Clean the user input
    cleaned = clean_text(user_review)

    # Convert text to vector
    vector = cv.transform([cleaned])

    # Predict sentiment
    prediction = model.predict(vector)[0]

    # Return result
    if prediction == 1:
        return "Positive 😊"
    else:
        return "Negative 😞"


In [ ]:
import pickle

# Save the trained model
pickle.dump(model, open("sentiment_model.pkl", "wb"))

# Save the vectorizer
pickle.dump(cv, open("vectorizer.pkl", "wb"))